# Day 21: Qdrant Performance Benchmarking
## Measuring Search Latency Across Scales

Welcome to Day 21 of the 90-Day AI Engineering Mastery curriculum. Today, we focus on a critical production engineering skill: performance benchmarking. We will measure how search latency in Qdrant scales when we move from 1,000 to 10,000 vectors.

### Core Theory (Just-in-Time)

**Why Benchmark?**
In production, vector databases like Qdrant need to handle rapid similarity searches across millions of vectors. While HNSW (Hierarchical Navigable Small World) graphs — the underlying indexing algorithm in Qdrant — offer sub-linear search time (typically `O(log N)`), latency still grows as the dataset size increases. 

**How does Qdrant handle this?**
Qdrant builds an HNSW index to quickly approximate nearest neighbors. The time it takes to search depends on:
1. **Dataset Size (N):** The number of vectors in the collection.
2. **Vector Dimensionality (D):** The size of each vector (e.g., 1536 for OpenAI embeddings).
3. **Search Parameters (`hnsw_ef`, `limit`):** Higher `limit` (top K results) or `hnsw_ef` (search accuracy parameter) increases latency.

**AI Security & Production Implications:**
- **PII Leakage in Payloads:** When benchmarking or storing vectors, be cautious about including Personally Identifiable Information (PII) in metadata payloads. Ensure data is anonymized or redacted *before* insertion into the vector DB.
- **Graceful Fallbacks:** If the vector database is unreachable or query latency spikes beyond the SLA, your application should catch the timeout/error and fallback to a local cache, BM25 keyword search, or return a polite degradation message (e.g., "Search temporarily unavailable").

### Common Pitfalls in Production
1. **Ignoring Indexing Time:** HNSW indexes take time to build. In this benchmark, we focus on *search* latency, but inserting 10,000 vectors might take noticeably longer than 1,000.
2. **Over-provisioning Dimensions:** Higher dimensions increase memory usage and distance calculation time. Only use the dimensionality you actually need.
3. **Benchmarking with In-Memory vs. Disk:** Memory mode (`:memory:`) is fast but doesn't reflect the I/O bottleneck you might hit when vectors are stored on disk in a real cluster.


### Code Implementation: Basic
Isolating the core concept with minimal boilerplate. We set up an in-memory client and perform a quick search latency test.

In [1]:
import time
import uuid
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

# Basic benchmarking
DIMENSION = 128
client = QdrantClient(":memory:")

# Setup collection
if client.collection_exists(collection_name="basic_1k"):
    client.delete_collection(collection_name="basic_1k")

client.create_collection(
    collection_name="basic_1k",
    vectors_config=VectorParams(size=DIMENSION, distance=Distance.COSINE),
)

# Insert 1000 vectors
points = [
    PointStruct(id=str(uuid.uuid4()), vector=np.random.rand(DIMENSION).astype(np.float32).tolist(), payload={})
    for _ in range(1000)
]
client.upload_points(collection_name="basic_1k", points=points)

# Benchmark
start_time = time.perf_counter()
for _ in range(50):
    client.query_points(
        collection_name="basic_1k",
        query=np.random.rand(DIMENSION).astype(np.float32).tolist(),
        limit=10
    )
latency = (time.perf_counter() - start_time) / 50 * 1000
print(f"Basic 1K Latency: {latency:.4f} ms")

Basic 1K Latency: 1.1517 ms


### Code Implementation: Medium
Here, we introduce clean Object-Oriented Programming (OOP) and state management. The benchmark logic is encapsulated in a class.

In [2]:
class QdrantBenchmark:
    def __init__(self, dimension: int):
        self.dimension = dimension
        self.client = QdrantClient(":memory:")

    def setup_and_upload(self, collection_name: str, num_points: int) -> None:
        if self.client.collection_exists(collection_name=collection_name):
            self.client.delete_collection(collection_name=collection_name)
        
        self.client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=self.dimension, distance=Distance.COSINE),
        )
        points = [
            PointStruct(id=str(uuid.uuid4()), vector=np.random.rand(self.dimension).astype(np.float32).tolist(), payload={})
            for _ in range(num_points)
        ]
        self.client.upload_points(collection_name=collection_name, points=points)

    def measure_latency(self, collection_name: str, num_queries: int = 50) -> float:
        total_time = 0.0
        for _ in range(num_queries):
            query = np.random.rand(self.dimension).astype(np.float32).tolist()
            start_time = time.perf_counter()
            self.client.query_points(collection_name=collection_name, query=query, limit=10)
            total_time += time.perf_counter() - start_time
        return (total_time / num_queries) * 1000

benchmark = QdrantBenchmark(dimension=128)
benchmark.setup_and_upload("medium_1k", 1000)
benchmark.setup_and_upload("medium_10k", 10000)

print(f"Medium 1K Latency: {benchmark.measure_latency('medium_1k'):.4f} ms")
print(f"Medium 10K Latency: {benchmark.measure_latency('medium_10k'):.4f} ms")

Medium 1K Latency: 0.9655 ms


Medium 10K Latency: 4.2384 ms


### Code Implementation: Advanced
Production-grade implementation with strict type hinting, docstrings, error handling, exact import syntax, and a graceful fallback mechanism to ensure AI Security and reliability.

In [3]:
import logging
from typing import List, Dict, Any, Optional
import time
import uuid
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ProductionQdrantBenchmark:
    """
    Production-grade benchmark class demonstrating robust state management, 
    error handling, and graceful fallbacks.
    """
    def __init__(self, dimension: int, location: str = ":memory:"):
        self.dimension = dimension
        try:
            self.client = QdrantClient(location)
            logger.info("QdrantClient initialized successfully.")
        except Exception as e:
            logger.error(f"Failed to initialize Qdrant client: {e}")
            raise

    def setup_collection(self, collection_name: str) -> None:
        """Creates a Qdrant collection securely."""
        try:
            if self.client.collection_exists(collection_name=collection_name):
                self.client.delete_collection(collection_name=collection_name)
            self.client.create_collection(
                collection_name=collection_name,
                vectors_config=VectorParams(size=self.dimension, distance=Distance.COSINE),
            )
            logger.info(f"Collection {collection_name} setup completed.")
        except Exception as e:
            logger.error(f"Error setting up collection: {e}")
            raise

    def secure_upload(self, collection_name: str, num_points: int) -> None:
        """
        Uploads data with simulated PII redaction and batching.
        """
        points: List[PointStruct] = []
        for i in range(num_points):
            # AI Security: Redacting PII before vector insertion
            safe_payload: Dict[str, Any] = {"metadata": f"Doc {i}", "pii_redacted": True}
            
            points.append(
                PointStruct(
                    id=str(uuid.uuid4()), 
                    vector=np.random.rand(self.dimension).astype(np.float32).tolist(), 
                    payload=safe_payload
                )
            )
        try:
            # Note: in real production, upload in batches
            self.client.upload_points(collection_name=collection_name, points=points)
            logger.info(f"Uploaded {num_points} points securely.")
        except Exception as e:
            logger.error(f"Failed to upload points: {e}")

    def safe_query(self, collection_name: str, query: List[float], limit: int = 10) -> Optional[List[Any]]:
        """Queries the collection with a graceful fallback."""
        try:
            # Fallback mechanism if query takes too long or fails
            return self.client.query_points(collection_name=collection_name, query=query, limit=limit)
        except Exception as e:
            logger.warning(f"Vector search failed ({e}). Graceful fallback: Returning empty/cached results.")
            return []

    def run_benchmark(self, collection_name: str, num_queries: int = 50) -> float:
        """Executes the benchmark and measures latency."""
        total_time = 0.0
        for _ in range(num_queries):
            query = np.random.rand(self.dimension).astype(np.float32).tolist()
            start_time = time.perf_counter()
            self.safe_query(collection_name, query)
            total_time += time.perf_counter() - start_time
        avg_latency = (total_time / num_queries) * 1000
        logger.info(f"Benchmark for {collection_name}: {avg_latency:.4f} ms")
        return avg_latency

# Execution
prod_benchmark = ProductionQdrantBenchmark(dimension=128)
prod_benchmark.setup_collection("prod_10k")
prod_benchmark.secure_upload("prod_10k", 10000)
prod_latency = prod_benchmark.run_benchmark("prod_10k")
print(f"Advanced 10K Latency: {prod_latency:.4f} ms")

INFO:__main__:QdrantClient initialized successfully.


INFO:__main__:Collection prod_10k setup completed.


INFO:__main__:Uploaded 10000 points securely.


INFO:__main__:Benchmark for prod_10k: 4.3683 ms


Advanced 10K Latency: 4.3683 ms


### Practical Lab / Homework

**Task:** 
Now that you have seen how collection size affects latency, your task is to measure how the `limit` parameter (the number of requested nearest neighbors) impacts search time.

Using the `prod_10k` collection created above, write a function that benchmarks the average search latency for different values of `limit`: `[10, 50, 100, 500]`. 
Print the average latency for each limit.

**Homework Requirement:** Record a brief async video walkthrough (e.g., using Loom) explaining your design decisions, how you managed state/errors, and how you observe latency scaling.

**Reference Implementation:**
*(Try to write it yourself first, then review the implementation below)*

In [4]:
def benchmark_limit(benchmark_client: ProductionQdrantBenchmark, collection_name: str, limits: List[int], num_queries: int = 50) -> None:
    """
    Benchmarks search latency across different limit values (top-k).
    """
    print(f"\nBenchmarking limits on {collection_name}:")
    for limit in limits:
        total_time = 0.0
        for _ in range(num_queries):
            query = np.random.rand(benchmark_client.dimension).astype(np.float32).tolist()
            
            start_time = time.perf_counter()
            try:
                benchmark_client.safe_query(
                    collection_name=collection_name,
                    query=query,
                    limit=limit
                )
            except Exception:
                pass # Handled in safe_query
            total_time += (time.perf_counter() - start_time)
            
        avg_latency_ms = (total_time / num_queries) * 1000
        print(f"  Limit = {limit:<4} | Avg Latency: {avg_latency_ms:.4f} ms")

# Execute the lab benchmark
limits_to_test = [10, 50, 100, 500]
benchmark_limit(prod_benchmark, "prod_10k", limits_to_test)



Benchmarking limits on prod_10k:


  Limit = 10   | Avg Latency: 4.2149 ms


  Limit = 50   | Avg Latency: 4.8790 ms


  Limit = 100  | Avg Latency: 5.8935 ms


  Limit = 500  | Avg Latency: 13.7568 ms


### Reference Links
- [Qdrant Official Documentation - Indexing](https://qdrant.tech/documentation/concepts/indexing/)
- [HNSW Algorithm Explained](https://arxiv.org/abs/1603.09320)
- [OWASP - Machine Learning Security Top 10](https://owasp.org/www-project-machine-learning-security-top-10/)
